# BƯỚC 3: TIỀN XỬ LÝ, LÀM SẠCH VÀ KHỬ TRÙNG LẶP DỮ LIỆU (DATA CLEANING & DEDUPLICATION)
## DỰ ÁN: IT JOB MARKET INTELLIGENCE
---
**Mục tiêu của Notebook:**
Xử lý toàn bộ các vấn đề chất lượng dữ liệu đã phát hiện ở Bước 2 qua 13 công đoạn làm sạch chuyên sâu:
- Chuẩn hóa văn bản Tiêu đề, Tên công ty, Mô tả công việc.
- Bóc tách và chuẩn hóa kỹ năng yêu cầu / ưu tiên (Canonical Skills Map).
- Bóc tách số năm kinh nghiệm (exp_min, exp_max) và chuẩn hóa mức lương về VNĐ/tháng (salary_min, salary_max).
- Chuẩn hóa địa điểm, hình thức làm việc, hạn nộp hồ sơ.
- **Khử trùng lặp thực thể (Entity Deduplication)** theo Công ty + Chức danh.
- Xuất tập dữ liệu sạch cuối cùng ra data/notebook/content_base/jobs_clean.csv phục vụ cho mô hình gợi ý.

## 0. Nạp dữ liệu thô ban đầu từ Bronze

In [1]:
# 1. Tìm đường dẫn gốc dự án (hoạt động đúng từ bất kỳ thư mục nào)
import os
import glob
import json
import re
from datetime import datetime
import pandas as pd

current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
project_root = current_dir
while not os.path.exists(os.path.join(project_root, 'data')) and os.path.dirname(project_root) != project_root:
    project_root = os.path.dirname(project_root)

# 2. Nạp bản đồ logo từ listings raw để bổ trợ logo công ty
url_to_logo = {}
comp_to_logo = {}
for p in glob.glob(os.path.join(project_root, 'data', 'raw', '*', 'listings', '*', '*.json')):
    try:
        with open(p, 'r', encoding='utf-8') as f:
            items = json.load(f)
        for it in items:
            u = (it.get('job_url') or it.get('detail_url') or it.get('url') or '').strip()
            lg = (it.get('company_logo_url') or it.get('companyLogo') or it.get('logo') or it.get('image_logo') or '').strip()
            cp = (it.get('company_name') or it.get('company_name_raw') or it.get('company') or '').strip().lower()
            if lg:
                if u and u not in url_to_logo:
                    url_to_logo[u] = lg
                if cp and cp not in comp_to_logo:
                    comp_to_logo[cp] = lg
    except Exception:
        pass

# 3. Nạp dữ liệu Master Bronze hiện có
bronze_file = os.path.join(project_root, 'data', 'bronze', 'bronze_details_valid.json')
existing_records = {}
if os.path.exists(bronze_file):
    with open(bronze_file, 'r', encoding='utf-8') as f:
        for item in json.load(f):
            jid = item.get('detail_id') or f"{item.get('source', '')}_{item.get('source_job_id', '')}"
            existing_records[jid] = item
print(f" Dữ liệu Bronze hiện có: {len(existing_records):,} bản ghi.")

# 4. Tự động quét toàn bộ thư mục data/raw/ để cập nhật thêm các tin mới cào
raw_root = os.path.join(project_root, 'data', 'raw')
newly_added = 0
if os.path.exists(raw_root):
    for source in ['topcv', 'topdev', 'vietnamworks']:
        details_dir = os.path.join(raw_root, source, 'details')
        if not os.path.isdir(details_dir):
            continue
        for root, _, files in os.walk(details_dir):
            for fname in files:
                if fname.endswith('.json'):
                    try:
                        with open(os.path.join(root, fname), 'r', encoding='utf-8') as rf:
                            data = json.load(rf)
                            if isinstance(data, list):
                                for it in data:
                                    jid = it.get('detail_id') or f"{it.get('source', '')}_{it.get('source_job_id', '')}"
                                    if jid not in existing_records:
                                        existing_records[jid] = it
                                        newly_added += 1
                    except Exception:
                        pass

if newly_added > 0:
    print(f" Phát hiện {newly_added:,} tin tuyển dụng mới từ data/raw/! Đang cập nhật Master Bronze...")
    with open(bronze_file, 'w', encoding='utf-8') as f:
        json.dump(list(existing_records.values()), f, ensure_ascii=False, indent=2)
    print(f" Đã đồng bộ Master Bronze ({len(existing_records):,} bản ghi).")
else:
    print(" Kho dữ liệu đã ở trạng thái mới nhất, sẵn sàng tiền xử lý.")

# 5. Trích xuất bảng dữ liệu phục vụ bài toán Recommendation (đầy đủ 18 thuộc tính)
records = []
for item in existing_records.values():
    desc_parts = [item.get(k, '') for k in ['overview_raw', 'responsibilities_raw', 'requirements_raw', 'benefits_raw'] if item.get(k)]
    j_url = (item.get('job_url') or '').strip()
    cp_name = (item.get('company_name_raw') or '').strip()
    logo = (item.get('company_logo_url') or url_to_logo.get(j_url) or comp_to_logo.get(cp_name.lower()) or '').strip()
    if logo:
        logo = re.sub(r'^https?://cdn-new\.topcv\.vn/unsafe/(https?://)', r'\1', logo).strip()
    records.append({
        'job_id': item.get('detail_id') or f"{item.get('source', '')}_{item.get('source_job_id', '')}",
        'company_name_raw': cp_name,
        'title_raw': (item.get('title_raw') or '').strip(),
        'description_raw': '\n\n'.join(desc_parts),
        'required_skills_raw': (item.get('required_skills_raw') or '').strip(),
        'preferred_skills_raw': (item.get('preferred_skills_raw') or '').strip(),
        'experience_raw': (item.get('experience_raw') or '').strip(),
        'location_raw': (item.get('location_raw') or item.get('company_location_raw') or '').strip(),
        'salary_raw': (item.get('salary_raw') or '').strip(),
        'employment_type_raw': (item.get('employment_type_raw') or '').strip(),
        'work_mode_raw': (item.get('work_mode_raw') or '').strip(),
        'deadline_raw': (item.get('deadline_raw') or '').strip(),
        'job_url': j_url,
        'company_logo_url': logo
    })

jobs = pd.DataFrame(records)
print(f" Khởi tạo DataFrame: {len(jobs):,} bản ghi x {jobs.shape[1]} thuộc tính.")
jobs.head(3)


 Dữ liệu Bronze hiện có: 5,491 bản ghi.
 Kho dữ liệu đã ở trạng thái mới nhất, sẵn sàng tiền xử lý.
 Khởi tạo DataFrame: 5,491 bản ghi x 14 thuộc tính.


,job_id,company_name_raw,title_raw,description_raw,required_skills_raw,preferred_skills_raw,experience_raw,location_raw,salary_raw,employment_type_raw,work_mode_raw,deadline_raw,job_url,company_logo_url
0,topcv_2314643,Công ty TNHH ZIGExN VeNtura,Full-Stack Web Engineer Intern/Fresher (AI Co...,COMPANY INTRODUCTION\nAbout our parent company...,"Không yêu cầu kinh nghiệm chuyên môn, Fullstac...",,Không yêu cầu,"Floor 11, Vina Building, 131 Xo Viet Nghe Tinh...",Thoả thuận,Thực tập sinh,Onsite,2026-10-28T23:59:59+07:00,https://www.topcv.vn/viec-lam/full-stack-web-e...,https://static.topcv.vn/company_logos/67c94ea3...
1,topcv_2314547,Công ty TNHH Điện tử R&P,Kỹ Sư Hệ Thống Nhúng,"R&P là công ty công nghệ, hoạt động trên lĩnh ...","Không yêu cầu kinh nghiệm chuyên môn, Embedded...",,Không yêu cầu,"65 Lê Trung Nghĩa, Phường Bảy Hiền, Hồ Chí Minh",Thoả thuận,Nhân viên,Onsite,2026-10-28T23:59:59+07:00,https://www.topcv.vn/viec-lam/ky-su-he-thong-n...,https://static.topcv.vn/company_logos/cong-ty-...
2,topcv_2321616,CÔNG TY TNHH FOXSCORE,UI/UX Designer - Mobile App- [HCM] - Thu Nhập ...,FOXSCORE – NỀN TẢNG THỂ THAO THÔNG MINH\nRa đờ...,"2 năm kinh nghiệm chuyên môn, Product Owner/Pr...",,2 năm,"62/6-8-10 Trương Công Định, Phường Tân Bình, H...",15000000,Nhân viên,Onsite,2026-11-05T23:59:59+07:00,https://www.topcv.vn/viec-lam/ui-ux-designer-m...,https://static.topcv.vn/company_logos/ELdaFFf6...


## Bước 1: Làm sạch Tên công ty (`company_name_raw` ➔ `company_name`)

Loại bỏ các tiền tố/hậu tố pháp lý (*Công ty TNHH, Cổ phần, JSC, LLC, Co., Ltd...*) để đồng nhất tên thương hiệu.

In [2]:
def clean_company_name(name):
    if not name or pd.isna(name):
        return "Không rõ"
    s = str(name).strip()
    s = re.sub(r'^(?:công ty|cty)?\s*(?:tnhh mtv|tnhh|cổ phần|cp|tập đoàn)\s*', '', s, flags=re.IGNORECASE)
    s = re.sub(r'[\s,\-]+(?:tnhh|cổ phần|jsc|llc|inc|corp|co\.,?\s*ltd\.?|ltd\.?)$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'[\s,\-]+(?:chi nhánh|vietnam|việt nam)$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+', ' ', s).strip(' -,/:;')
    return s if s else str(name).strip()

jobs['company_name'] = jobs['company_name_raw'].apply(clean_company_name)

print("=== BƯỚC 1: TRƯỚC VÀ SAU KHI LÀM SẠCH COMPANY NAME ===")
jobs[['company_name_raw', 'company_name']].head(5)


=== BƯỚC 1: TRƯỚC VÀ SAU KHI LÀM SẠCH COMPANY NAME ===


,company_name_raw,company_name
0,Công ty TNHH ZIGExN VeNtura,ZIGExN VeNtura
1,Công ty TNHH Điện tử R&P,Điện tử R&P
2,CÔNG TY TNHH FOXSCORE,FOXSCORE
3,CÔNG TY CỔ PHẦN THƯƠNG MẠI QUỐC TẾ LB GLOBAL,THƯƠNG MẠI QUỐC TẾ LB GLOBAL
4,CÔNG TY TNHH VODAPLAY VIỆT NAM,VODAPLAY


## Bước 2: Làm sạch Tiêu đề việc làm (`title_raw` ➔ `title`)

Loại bỏ nội dung trong ngoặc vuông `[...]`, địa điểm `(HCM)`, tiền tố tuyển gấp, và các đoạn quảng cáo lương ở đuôi.

In [3]:
def clean_job_title(title):
    if not title or pd.isna(title):
        return ""
    t = str(title).strip()
    t = re.sub(r'\[.*?\]', '', t)
    t = re.sub(r'\((?:hà nội|hcm|đà nẵng|hồ chí minh|toàn quốc)\)', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^\s*(?:tuyển gấp|tuyển dụng|gấp:?)\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'(?:-|\|)?\s*(?:thu nhập|lương|up to|upto|lên đến|tới)\s*.*$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'(?:-|\|)?\s*(?:hấp dẫn|deal|thỏa thuận|cạnh tranh)\s*$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s+', ' ', t).strip(' -|/,:;')
    return t

jobs['title'] = jobs['title_raw'].apply(clean_job_title)

print("=== BƯỚC 2: TRƯỚC VÀ SAU KHI LÀM SẠCH TITLE ===")
jobs[['title_raw', 'title']].head(5)


=== BƯỚC 2: TRƯỚC VÀ SAU KHI LÀM SẠCH TITLE ===


,title_raw,title
0,Full-Stack Web Engineer Intern/Fresher (AI Co...,Full-Stack Web Engineer Intern/Fresher (AI Cod...
1,Kỹ Sư Hệ Thống Nhúng,Kỹ Sư Hệ Thống Nhúng
2,UI/UX Designer - Mobile App- [HCM] - Thu Nhập ...,UI/UX Designer - Mobile App
3,Business Analyst - Data Hệ Thống Phân Phối - C...,Business Analyst - Data Hệ Thống Phân Phối - C...
4,Junior QA Tester (Manual Testing) Upto 16M,Junior QA Tester (Manual Testing)


## Bước 3: Làm sạch Văn bản Mô tả (`description_raw` ➔ `description_clean`)

Gỡ thẻ HTML, giải mã thực thể HTML (`&nbsp;`, `&amp;`), đồng bộ bullet points về `- `, và bỏ các dòng trống thừa.

In [4]:
def clean_html_description(text):
    if not text or pd.isna(text):
        return ""
    s = str(text)
    s = s.replace('&nbsp;', ' ').replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"')
    s = re.sub(r'<(?:br|/p|/li|/div|/h\d)[^>]*>', '\n', s, flags=re.IGNORECASE)
    s = re.sub(r'<[^>]+>', ' ', s)
    s = re.sub(r'^[ \t]*[-\*\+\–•][ \t]*', '- ', s, flags=re.MULTILINE)
    lines = [re.sub(r'[ \t]+', ' ', l).strip() for l in s.split('\n')]
    return '\n'.join([l for l in lines if l])

jobs['description_clean'] = jobs['description_raw'].apply(clean_html_description)

print("=== BƯỚC 3: MẪU DESCRIPTION_CLEAN SAU KHI LÀM SẠCH (400 KÝ TỰ ĐẦU) ===")
print(jobs['description_clean'].iloc[0][:400] + "...")
jobs[['title', 'description_clean']].head(3)


=== BƯỚC 3: MẪU DESCRIPTION_CLEAN SAU KHI LÀM SẠCH (400 KÝ TỰ ĐẦU) ===
COMPANY INTRODUCTION
About our parent company: ZIGExN Co.,Ltd
- ZIGEXN is a leading life media platform company in Japan, offering over 40 services in various life event domains such as jobs, housing, cars, and travel.
- ZIGExN, the company's flagship, received the Japan Technology Fast 50 award from Deloitte in 2015 and has been recognized as a Great Place to Work in Japan from 2011 to 2016.
- ZI...


,title,description_clean
0,Full-Stack Web Engineer Intern/Fresher (AI Cod...,COMPANY INTRODUCTION\nAbout our parent company...
1,Kỹ Sư Hệ Thống Nhúng,"R&P là công ty công nghệ, hoạt động trên lĩnh ..."
2,UI/UX Designer - Mobile App,FOXSCORE – NỀN TẢNG THỂ THAO THÔNG MINH\nRa đờ...


## Bước 4: Làm sạch Kỹ năng (`required_skills_raw` ➔ `required_skills` + `job_tags`)

**Vấn đề phát hiện khi kiểm tra dữ liệu:** trường `required_skills_raw` của TopCV/VietnamWorks **không chỉ chứa kỹ năng**. Nó trộn lẫn thẻ phân loại của sàn:

| Loại thẻ | Ví dụ | Xử lý |
|---|---|---|
| Kỹ năng công nghệ | `Python`, `SQL`, `Docker`, `CI/CD` | ➔ `required_skills` |
| Ngành / lĩnh vực / chức danh / ngoại ngữ | `It - Phần Mềm`, `Ngân Hàng`, `B2B`, `Backend Developer`, `Tiếng Anh Đọc Hiểu` | ➔ `job_tags` (giữ làm ngữ cảnh, **không** tính là kỹ năng) |
| Rác (phúc lợi, kinh nghiệm, học vấn) | `Xăng Xe`, `2 năm kinh nghiệm chuyên môn`, `Đại học trở lên` | ➔ loại bỏ |

Nếu không tách, thẻ `It - Phần Mềm` (xuất hiện ở ~31% tin) trở thành "kỹ năng" phổ biến nhất, làm sai lệch cả điểm khớp kỹ năng lẫn vector TF-IDF của hệ gợi ý (kỹ năng được nhân trọng số ×4).

**Cách làm:** một thẻ là *kỹ năng* khi nằm trong từ điển công nghệ (`TECH_LOOKUP`). Ngoài ra bổ sung kỹ năng quét từ phần mô tả JD, dùng regex có biên từ an toàn (nhận đúng `C++`, `C#`, `.NET`, `CI/CD`; bỏ qua các khóa mơ hồ như `go`, `cv`, `ml`).

In [ ]:
import re

# 1. TỪ ĐIỂN KỸ NĂNG CÔNG NGHỆ: tên viết tắt / biến thể ➔ tên chuẩn (python3 ➔ Python, k8s ➔ Kubernetes...)
CANONICAL_SKILL_MAP = {
    'python': 'Python', 'py': 'Python', 'python3': 'Python', 'java': 'Java', 'java core': 'Java',
    'javascript': 'JavaScript', 'js': 'JavaScript', 'typescript': 'TypeScript', 'ts': 'TypeScript',
    'c++': 'C++', 'cpp': 'C++', 'c#': 'C#', 'csharp': 'C#', 'php': 'PHP', 'golang': 'Golang', 'go': 'Golang',
    'ruby': 'Ruby', 'rust': 'Rust', 'scala': 'Scala', 'kotlin': 'Kotlin', 'swift': 'Swift', 'dart': 'Dart',
    'r': 'R', 'bash': 'Bash', 'shell': 'Shell', 'powershell': 'PowerShell', 'sql': 'SQL', 't-sql': 'T-SQL',
    'pl/sql': 'PL/SQL', 'power bi': 'Power BI', 'powerbi': 'Power BI', 'pbi': 'Power BI',
    'tableau': 'Tableau', 'excel': 'Excel', 'vba': 'VBA', 'macro': 'VBA', 'dax': 'DAX',
    'power query': 'Power Query', 'powerquery': 'Power Query', 'looker': 'Looker', 'looker studio': 'Looker',
    'metabase': 'Metabase', 'google data studio': 'Looker', 'qlik': 'Qlik', 'qliksense': 'Qlik',
    'qlikview': 'Qlik', 'ssis': 'SSIS', 'ssrs': 'SSRS', 'ssas': 'SSAS',
    'google analytics': 'Google Analytics', 'ga4': 'Google Analytics', 'a/b testing': 'A/B Testing',
    'ab testing': 'A/B Testing', 'etl': 'ETL', 'elt': 'ELT', 'data warehouse': 'Data Warehouse',
    'dwh': 'Data Warehouse', 'data lake': 'Data Lake', 'data modeling': 'Data Modeling',
    'data pipeline': 'Data Pipeline', 'bigquery': 'BigQuery', 'snowflake': 'Snowflake',
    'spark': 'Apache Spark', 'pyspark': 'PySpark', 'apache spark': 'Apache Spark', 'hadoop': 'Hadoop',
    'hive': 'Hive', 'airflow': 'Apache Airflow', 'apache airflow': 'Apache Airflow', 'dbt': 'dbt',
    'kafka': 'Apache Kafka', 'apache kafka': 'Apache Kafka', 'spark streaming': 'Apache Spark',
    'machine learning': 'Machine Learning', 'ml': 'Machine Learning', 'deep learning': 'Deep Learning',
    'dl': 'Deep Learning', 'nlp': 'NLP', 'natural language processing': 'NLP',
    'computer vision': 'Computer Vision', 'cv': 'Computer Vision', 'pandas': 'Pandas', 'numpy': 'NumPy',
    'scikit-learn': 'Scikit-Learn', 'sklearn': 'Scikit-Learn', 'tensorflow': 'TensorFlow',
    'pytorch': 'PyTorch', 'keras': 'Keras', 'generative ai': 'Generative AI', 'genai': 'Generative AI',
    'llm': 'LLM', 'statistics': 'Statistics', 'spring boot': 'Spring Boot', 'springboot': 'Spring Boot',
    'spring': 'Spring Boot', 'django': 'Django', 'fastapi': 'FastAPI', 'flask': 'Flask',
    'node.js': 'Node.js', 'nodejs': 'Node.js', 'node': 'Node.js', 'express.js': 'Express.js',
    'expressjs': 'Express.js', 'express': 'Express.js', 'nestjs': 'NestJS', 'nest.js': 'NestJS',
    '.net': '.NET', 'dotnet': '.NET', '.net core': '.NET', 'asp.net': '.NET', 'laravel': 'Laravel',
    'ruby on rails': 'Ruby on Rails', 'rails': 'Ruby on Rails', 'gin': 'Gin', 'graphql': 'GraphQL',
    'rest api': 'REST API', 'restful': 'REST API', 'restful api': 'REST API', 'grpc': 'gRPC',
    'microservices': 'Microservices', 'microservice': 'Microservices', 'react': 'React', 'reactjs': 'React',
    'react.js': 'React', 'react native': 'React Native', 'reactnative': 'React Native', 'vue': 'Vue',
    'vuejs': 'Vue', 'vue.js': 'Vue', 'angular': 'Angular', 'angularjs': 'Angular', 'next.js': 'Next.js',
    'nextjs': 'Next.js', 'nuxt.js': 'Nuxt.js', 'nuxtjs': 'Nuxt.js', 'svelte': 'Svelte', 'flutter': 'Flutter',
    'android': 'Android', 'ios': 'iOS', 'html': 'HTML/CSS', 'html5': 'HTML/CSS', 'css': 'HTML/CSS',
    'css3': 'HTML/CSS', 'tailwind': 'TailwindCSS', 'tailwindcss': 'TailwindCSS', 'bootstrap': 'Bootstrap',
    'redux': 'Redux', 'postgresql': 'PostgreSQL', 'postgres': 'PostgreSQL', 'psql': 'PostgreSQL',
    'mysql': 'MySQL', 'sql server': 'SQL Server', 'mssql': 'SQL Server', 'oracle': 'Oracle Database',
    'mongodb': 'MongoDB', 'mongo': 'MongoDB', 'redis': 'Redis', 'elasticsearch': 'Elasticsearch',
    'elastic': 'Elasticsearch', 'cassandra': 'Cassandra', 'dynamodb': 'DynamoDB', 'sqlite': 'SQLite',
    'mariadb': 'MariaDB', 'firebase': 'Firebase', 'neo4j': 'Neo4j', 'aws': 'AWS',
    'amazon web services': 'AWS', 'azure': 'Azure', 'microsoft azure': 'Azure', 'gcp': 'GCP',
    'google cloud': 'GCP', 'docker': 'Docker', 'kubernetes': 'Kubernetes', 'k8s': 'Kubernetes',
    'ci/cd': 'CI/CD', 'cicd': 'CI/CD', 'git': 'Git', 'github': 'Git', 'gitlab': 'Git', 'linux': 'Linux',
    'terraform': 'Terraform', 'ansible': 'Ansible', 'jenkins': 'Jenkins', 'nginx': 'Nginx',
    'prometheus': 'Prometheus', 'grafana': 'Grafana', 'rabbitmq': 'RabbitMQ',
    'manual testing': 'Manual Testing', 'manual test': 'Manual Testing',
    'automation testing': 'Automation Testing', 'automation test': 'Automation Testing',
    'selenium': 'Selenium', 'cypress': 'Cypress', 'playwright': 'Playwright', 'postman': 'Postman',
    'jmeter': 'JMeter', 'unit test': 'Unit Testing', 'unit testing': 'Unit Testing', 'appium': 'Appium',
    'jira': 'Jira', 'confluence': 'Confluence', 'figma': 'Figma', 'photoshop': 'Photoshop', 'agile': 'Agile',
    'scrum': 'Scrum', 'oop': 'OOP', 'system design': 'System Design', 'an ninh mạng': 'An Ninh Mạng',
    'cybersecurity': 'An Ninh Mạng',
}

# 2. Khái niệm công nghệ hay gặp trong thẻ của sàn nhưng chưa có trong từ điển trên
TECH_EXTRA = {
    'data analysis': 'Data Analysis', 'data analytics': 'Data Analysis', 'phân tích dữ liệu': 'Data Analysis',
    'data science': 'Data Science', 'big data': 'Big Data', 'business intelligence': 'Business Intelligence',
    'data management': 'Data Management', 'data governance': 'Data Governance',
    'cloud': 'Cloud', 'điện toán đám mây (cloud)': 'Cloud', 'devops': 'DevOps', 'api': 'API', 'api testing': 'API Testing',
    'erp': 'ERP', 'sap': 'SAP', 'scada': 'SCADA', 'vmware': 'VMware', 'hosting': 'Hosting', 'server': 'Server',
    'software testing': 'Software Testing', 'kiểm thử phần mềm': 'Software Testing', 'qa': 'QA',
    'information security': 'An Ninh Mạng', 'it security': 'An Ninh Mạng', 'an toàn thông tin': 'An Ninh Mạng',
    'network security': 'Network Security',
    'lập trình nhúng': 'Embedded', 'embedded': 'Embedded', 'network': 'Networking', 'quản trị mạng': 'Networking',
    'system administration': 'System Administration', 'quản trị hệ thống': 'System Administration',
    'agile scrum': 'Agile', 'ux design': 'UX Design', 'ui': 'UI Design',
    'ms sql': 'SQL Server', 'python programming': 'Python', 'java spring boot': 'Spring Boot',
    'ci': 'CI/CD', 'cd': 'CI/CD', 'cd methodologies': 'CI/CD', 'ai': 'AI', 'blockchain': 'Blockchain',
}
TECH_LOOKUP = {**CANONICAL_SKILL_MAP, **TECH_EXTRA}

# 3. Khóa quá mơ hồ khi quét VĂN BẢN TỰ DO (vd 'go' trong "go to market", 'cv' trong "gửi CV", 'ml' viết tắt)
#    ➔ chỉ chấp nhận chúng khi xuất hiện như một THẺ rõ ràng, không quét từ mô tả.
AMBIGUOUS_FREE_TEXT_KEYS = {'cv', 'go', 'r', 'ml', 'dl', 'py', 'ts', 'js', 'macro', 'gin', 'spring', 'express',
                            'node', 'shell', 'elastic', 'ai', 'ui', 'ci', 'cd', 'api', 'qa', 'server', 'cloud', 'network', 'sap'}

# 4. Mẫu nhận diện thẻ RÁC (phúc lợi, kinh nghiệm, học vấn, nhân thân...)
JUNK_PATTERN = re.compile(
    r'bảo hiểm|thưởng|bonus|du lịch|team\s*building|khám sức khỏe|phụ cấp|xăng|tai nghe|cước|điện thoại|phương tiện|'
    r'giờ hành chính|giọng|phát âm|diễn đạt|kinh nghiệm|trở lên|đại học|cao đẳng|trung cấp|tuổi|xem tất cả|'
    r'toàn thời gian|bán thời gian|thực tập|signing', re.IGNORECASE)

# 5. Regex quét kỹ năng trong văn bản. Dùng lookaround thay cho \b để nhận đúng C++, C#, .NET, CI/CD
TECH_PATTERNS = [
    (re.compile(r'(?<![\w+#.])' + re.escape(key) + r'(?![\w+#])'), canonical)
    for key, canonical in TECH_LOOKUP.items() if key not in AMBIGUOUS_FREE_TEXT_KEYS
]


def classify_tag(tag):
    """Phân loại một thẻ thô: ('skill', tên chuẩn) | ('tag', tên giữ nguyên) | ('junk', None)."""
    key = re.sub(r'\s+', ' ', tag).strip().lower()
    if key in TECH_LOOKUP:
        return 'skill', TECH_LOOKUP[key]
    if JUNK_PATTERN.search(key):
        return 'junk', None
    return 'tag', re.sub(r'\s+', ' ', tag).strip()


def extract_skills_and_tags(row):
    skills, tags, seen_s, seen_t = [], [], set(), set()

    # (a) Tách chuỗi thẻ thô. KHÔNG tách theo dấu "/" để giữ nguyên CI/CD, A/B Testing
    for part in re.split(r'[,;|•\n]', str(row.get('required_skills_raw', ''))):
        item = part.strip()
        if len(item) <= 1 or len(item) > 60:
            continue
        kind, name = classify_tag(item)
        if kind == 'skill' and name.lower() not in seen_s:
            seen_s.add(name.lower()); skills.append(name)
        elif kind == 'tag' and name.lower() not in seen_t:
            seen_t.add(name.lower()); tags.append(name)

    # (b) Bổ sung kỹ năng công nghệ quét từ mô tả JD
    desc = str(row.get('description_clean', '')).lower()
    for pattern, canonical in TECH_PATTERNS:
        if canonical.lower() not in seen_s and pattern.search(desc):
            seen_s.add(canonical.lower()); skills.append(canonical)
    return skills, tags


skill_tag = jobs.apply(extract_skills_and_tags, axis=1)
jobs['required_skills'] = skill_tag.apply(lambda x: x[0])
jobs['job_tags'] = skill_tag.apply(lambda x: x[1])

# 6. Thống kê trước / sau để kiểm chứng
from collections import Counter
raw_tokens = Counter(t.strip() for s in jobs['required_skills_raw'] for t in str(s).split(',') if t.strip())
new_tokens = Counter(s for lst in jobs['required_skills'] for s in lst)
n_skill = jobs['required_skills'].apply(len)
print("=== BƯỚC 4: KẾT QUẢ TÁCH KỸ NĂNG / THẺ ===")
print(f"- Thẻ thô phổ biến nhất       : {raw_tokens.most_common(5)}")
print(f"- Kỹ năng sau xử lý phổ biến  : {new_tokens.most_common(8)}")
print(f"- Số kỹ năng / tin (trung bình): {n_skill.mean():.1f} | tin không có kỹ năng nào: {(n_skill == 0).mean():.1%}")
jobs[['title', 'required_skills_raw', 'required_skills', 'job_tags']].head(5)

## Bước 5: Bóc tách Kỹ năng ưu tiên (`preferred_skills_raw` & `description_clean` ➔ `preferred_skills`)

Do trên các sàn trường này bị rỗng 100%, ta sẽ quét đoạn văn bản `description_clean` tìm các đoạn có chứa từ khóa *"Ưu tiên"*, *"Điểm cộng"*, *"Nice to have"* để trích xuất kỹ năng.

In [6]:
POPULAR_TECH_KEYWORDS = [
    'Docker', 'Kubernetes', 'AWS', 'GCP', 'Azure', 'CI/CD', 'Git', 'Redis', 'Kafka', 'GraphQL',
    'React', 'Vue', 'Angular', 'Node.js', 'Python', 'Java', 'Golang', 'TypeScript', 'Flutter',
    'PostgreSQL', 'MongoDB', 'MySQL', 'Elasticsearch', 'Spring Boot', 'Next.js', 'TailwindCSS'
]

def extract_preferred_skills(row):
    raw_pref = row['preferred_skills_raw']
    if raw_pref and pd.notnull(raw_pref) and str(raw_pref).strip():
        return clean_required_skills(raw_pref)
    
    desc = str(row['description_clean']).lower()
    pref_section = ""
    m = re.search(r'(?:ưu tiên|điểm cộng|nice to have|plus|yêu cầu khác)[\s:]*([^\.\n]+)', desc)
    if m:
        pref_section = m.group(1)
            
    found_skills = []
    req_skills_lower = [k.lower() for k in row['required_skills']]
    for tech in POPULAR_TECH_KEYWORDS:
        if tech.lower() in pref_section and tech.lower() not in req_skills_lower:
            found_skills.append(tech)
    return found_skills

jobs['preferred_skills'] = jobs.apply(extract_preferred_skills, axis=1)

print("=== BƯỚC 5: KẾT QUẢ BÓC TÁCH PREFERRED_SKILLS ===")
jobs[['title', 'required_skills', 'preferred_skills']].head(5)


=== BƯỚC 5: KẾT QUẢ BÓC TÁCH PREFERRED_SKILLS ===


,title,required_skills,preferred_skills
0,Full-Stack Web Engineer Intern/Fresher (AI Cod...,"[Fullstack Developer, It - Phần Mềm, Tiếng Anh...",[]
1,Kỹ Sư Hệ Thống Nhúng,"[Embedded Engineer, Lập Trình Nhúng, Điện, Điệ...",[]
2,UI/UX Designer - Mobile App,"[Product Owner, Product Manager, It - Phần Mềm...",[]
3,Business Analyst - Data Hệ Thống Phân Phối - C...,"[Vận hành sàn thương mại điện tử, Excel]",[]
4,Junior QA Tester (Manual Testing),"[Manual Tester, It - Phần Mềm, Manual Testing]",[]


## Bước 6: Bóc tách Yêu cầu kinh nghiệm (`experience_raw` ➔ `experience_min`, `experience_max`)

Quy đổi các chuỗi tiếng Việt thô (*"Không yêu cầu"*, *"1 - 3 năm"*, *"Trên 5 năm"*, *"6 tháng"*) thành số năm dạng số thực.

In [7]:
def parse_experience_range(exp_str):
    if not exp_str or pd.isna(exp_str):
        return None, None
    s = str(exp_str).strip().lower()
    if any(k in s for k in ['không yêu cầu', 'chưa có kinh nghiệm', 'không cần', 'fresher', 'intern']):
        return 0.0, 0.0
    m_range = re.search(r'(\d+(?:\.\d+)?)\s*[-–]\s*(\d+(?:\.\d+)?)\s*năm', s)
    if m_range:
        return float(m_range.group(1)), float(m_range.group(2))
    m_min = re.search(r'(?:trên|từ|tối thiểu|\+)\s*(\d+(?:\.\d+)?)\s*năm|(\d+(?:\.\d+)?)\s*(?:\+|năm trở lên)', s)
    if m_min:
        v = float(m_min.group(1) or m_min.group(2))
        return v, None
    m_max = re.search(r'dưới\s*(\d+(?:\.\d+)?)\s*năm', s)
    if m_max:
        return 0.0, float(m_max.group(1))
    m_month = re.search(r'(\d+)\s*tháng', s)
    if m_month:
        years = round(float(m_month.group(1)) / 12.0, 1)
        return years, years
    m_exact = re.search(r'(\d+(?:\.\d+)?)\s*năm', s)
    if m_exact:
        v = float(m_exact.group(1))
        return v, v
    return None, None

exp_res = jobs['experience_raw'].apply(parse_experience_range)
jobs['experience_min'] = exp_res.apply(lambda x: x[0])
jobs['experience_max'] = exp_res.apply(lambda x: x[1])

print("=== BƯỚC 6: TRƯỚC VÀ SAU KHI BÓC TÁCH KINH NGHIỆM ===")
jobs[['experience_raw', 'experience_min', 'experience_max']].head(5)


=== BƯỚC 6: TRƯỚC VÀ SAU KHI BÓC TÁCH KINH NGHIỆM ===


,experience_raw,experience_min,experience_max
0,Không yêu cầu,0.0,0.0
1,Không yêu cầu,0.0,0.0
2,2 năm,2.0,2.0
3,3 tháng,0.2,0.2
4,1 năm,1.0,1.0


## Bước 7: Làm sạch Địa điểm & Trích xuất Thành phố (`location_raw` ➔ `location`, `city`)

Giữ địa chỉ chi tiết làm sạch, đồng thời gán nhãn thành phố chuẩn hóa (`Hồ Chí Minh`, `Hà Nội`, `Đà Nẵng`, `Khác`, `Toàn quốc`) phục vụ bộ lọc tìm việc.

**Các lỗi của phiên bản cũ đã sửa:**
- Địa chỉ **không dấu** (`hanoi`, `ha noi`, `ho chi minh`) trước đây rơi vào nhóm `Khác` ➔ giờ **bỏ dấu tiếng Việt trước khi so khớp**.
- Khớp chuỗi con làm `hn` có thể xuất hiện nhầm trong từ khác ➔ giờ dùng **ranh giới từ** (`\b`).
- Địa chỉ ghi nhiều thành phố (`Hà Nội; Hồ Chí Minh`) ➔ lấy thành phố **xuất hiện đầu tiên** (thường là trụ sở chính).

In [ ]:
import unicodedata

def strip_accents(text):
    """Hạ chữ thường và bỏ dấu tiếng Việt (đ ➔ d): 'Hà Nội' ➔ 'ha noi'."""
    s = unicodedata.normalize('NFD', str(text).lower().replace('đ', 'd'))
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')

# Mỗi thành phố là một regex trên chuỗi đã bỏ dấu (gồm tên thành phố + các quận/huyện không nhập nhằng)
CITY_PATTERNS = {
    'Hồ Chí Minh': re.compile(r'\b(?:ho chi minh|tp\.?\s*hcm|tphcm|hcmc|hcm|sai gon|saigon|thu duc|binh thanh|tan binh|'
                              r'phu nhuan|go vap|tan phu|binh tan|nha be|quan\s*\d{1,2}|district\s*\d{1,2})\b'),
    'Hà Nội':      re.compile(r'\b(?:ha noi|hanoi|hn|cau giay|nam tu liem|bac tu liem|ba dinh|dong da|hai ba trung|'
                              r'long bien|ha dong|hoan kiem|tay ho|thanh xuan|me tri|my dinh)\b'),
    'Đà Nẵng':     re.compile(r'\b(?:da nang|danang|hai chau|son tra|ngu hanh son|lien chieu|thanh khe)\b'),
}
NATIONWIDE = re.compile(r'\b(?:toan quoc|remote|viet nam|vietnam)\b')

def extract_city(loc_str):
    if loc_str is None or not str(loc_str).strip() or str(loc_str).lower() == 'nan':
        return 'Toàn quốc'
    s = strip_accents(loc_str)
    best_city, best_pos = None, None
    for city, pattern in CITY_PATTERNS.items():
        m = pattern.search(s)
        if m and (best_pos is None or m.start() < best_pos):   # chọn thành phố xuất hiện sớm nhất
            best_city, best_pos = city, m.start()
    if best_city:
        return best_city
    if NATIONWIDE.search(s) and len(s.split()) <= 3:           # chỉ ghi "Remote / Việt Nam"
        return 'Toàn quốc'
    return 'Khác'

jobs['location'] = jobs['location_raw'].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip() if pd.notnull(x) else "")
jobs['city'] = jobs['location'].apply(extract_city)

print("=== BƯỚC 7: TRƯỚC VÀ SAU KHI LÀM SẠCH LOCATION & TRÍCH XUẤT CITY ===")
print(jobs['city'].value_counts().to_string())
jobs[['location_raw', 'location', 'city']].head(5)

## Bước 8: Chuẩn hóa Mức lương (`salary_raw` ➔ `salary_min`, `salary_max` theo VNĐ/tháng)

Quy đổi tiền tệ (USD ➔ VNĐ tỷ giá 25,400; Triệu đồng ➔ VNĐ nhân 1,000,000), phân tách cận dưới và cận trên.

In [9]:
EXCHANGE_RATE_USD = 25400

def parse_salary_vnd(sal_str):
    if not sal_str or pd.isna(sal_str):
        return None, None
    s = str(sal_str).strip().lower()
    if any(k in s for k in ['thương lượng', 'cạnh tranh', 'thoa thuan', 'thỏa thuận', 'negotiable', 'deal']):
        return None, None
    
    has_usd = ('$' in s or 'usd' in s) and not ('tr' in s or 'triệu' in s)
    nums = [float(n.replace(',', '')) for n in re.findall(r'(\d+(?:,\d+)*(?:\.\d+)?)', s)]
    if not nums:
        return None, None
        
    if has_usd:
        nums = [n * EXCHANGE_RATE_USD for n in nums]
    else:
        if any(k in s for k in ['tr', 'triệu', 'trieu', 'm']) or any(n < 500 for n in nums):
            nums = [n * 1_000_000 for n in nums]
            
    if len(nums) >= 2:
        return min(nums[0], nums[1]), max(nums[0], nums[1])
    elif len(nums) == 1:
        val = nums[0]
        if any(k in s for k in ['tới', 'up to', 'dưới', '<', 'tối đa']):
            return None, val
        elif any(k in s for k in ['từ', 'trên', '>=', 'tối thiểu', 'from', '>']):
            return val, None
        return val, val
    return None, None

sal_res = jobs['salary_raw'].apply(parse_salary_vnd)
jobs['salary_min'] = sal_res.apply(lambda x: x[0])
jobs['salary_max'] = sal_res.apply(lambda x: x[1])

print("=== BƯỚC 8: TRƯỚC VÀ SAU KHI CHUẨN HÓA LƯƠNG SANG VNĐ ===")
jobs[['salary_raw', 'salary_min', 'salary_max']].head(5)


=== BƯỚC 8: TRƯỚC VÀ SAU KHI CHUẨN HÓA LƯƠNG SANG VNĐ ===


,salary_raw,salary_min,salary_max
0,Thoả thuận,NaN,NaN
1,Thoả thuận,NaN,NaN
2,15000000,15000000.0,15000000.0
3,12000000,12000000.0,12000000.0
4,Tới 16 triệu,NaN,16000000.0


## Bước 9: Chuẩn hóa Hình thức làm việc (`employment_type_raw` ➔ `employment_type`)

Ánh xạ về 4 nhóm chuẩn: `Intern`, `Part-time`, `Contract`, `Full-time`.

In [10]:
def clean_employment_type(emp_str, title_str=""):
    c = (str(emp_str) + " " + str(title_str)).lower()
    if any(k in c for k in ['intern', 'thực tập', 'trainee']):
        return 'Intern'
    if any(k in c for k in ['part-time', 'bán thời gian', 'parttime']):
        return 'Part-time'
    if any(k in c for k in ['freelance', 'hợp đồng', 'contract']):
        return 'Contract'
    return 'Full-time'

jobs['employment_type'] = jobs.apply(lambda r: clean_employment_type(r['employment_type_raw'], r['title']), axis=1)

print("=== BƯỚC 9: TRƯỚC VÀ SAU KHI CHUẨN HÓA EMPLOYMENT_TYPE ===")
jobs[['employment_type_raw', 'employment_type']].head(5)


=== BƯỚC 9: TRƯỚC VÀ SAU KHI CHUẨN HÓA EMPLOYMENT_TYPE ===


,employment_type_raw,employment_type
0,Thực tập sinh,Intern
1,Nhân viên,Full-time
2,Nhân viên,Full-time
3,Nhân viên,Full-time
4,Nhân viên,Full-time


## Bước 10: Chuẩn hóa Mô hình làm việc (`work_mode_raw` ➔ `work_mode`)

Ánh xạ về 3 nhóm chuẩn duy nhất: `Onsite`, `Hybrid`, `Remote`.

In [11]:
def clean_work_mode(mode_str, loc_str=""):
    c = (str(mode_str) + ' ' + str(loc_str)).lower()
    if any(k in c for k in ['remote', 'từ xa', 'làm việc từ xa']):
        return 'Remote'
    if any(k in c for k in ['hybrid', 'linh hoạt']):
        return 'Hybrid'
    return 'Onsite'

jobs['work_mode'] = jobs.apply(lambda r: clean_work_mode(r['work_mode_raw'], r['location']), axis=1)

print("=== BƯỚC 10: TRƯỚC VÀ SAU KHI CHUẨN HÓA WORK_MODE ===")
jobs[['work_mode_raw', 'work_mode']].head(5)


=== BƯỚC 10: TRƯỚC VÀ SAU KHI CHUẨN HÓA WORK_MODE ===


,work_mode_raw,work_mode
0,Onsite,Onsite
1,Onsite,Onsite
2,Onsite,Onsite
3,Onsite,Onsite
4,Onsite,Onsite


## Bước 11: Chuẩn hóa Hạn nộp & Tính hiệu lực (`deadline_raw` ➔ `deadline`, `is_active`)

Cắt lấy ngày `YYYY-MM-DD` và so sánh với ngày hiện tại để xác định tin còn hiệu lực (`is_active = True/False`).

In [12]:
def parse_deadline(d_str):
    if not d_str or pd.isna(d_str):
        return None, True
    s = str(d_str).strip()
    try:
        if 'T' in s:
            dt = datetime.fromisoformat(s)
        else:
            dt = datetime.strptime(s[:10], '%Y-%m-%d')
        d_formatted = dt.strftime('%Y-%m-%d')
        is_act = dt.date() >= datetime.now().date()
        return d_formatted, is_act
    except Exception:
        return s[:10], True

dl_res = jobs['deadline_raw'].apply(parse_deadline)
jobs['deadline'] = dl_res.apply(lambda x: x[0])
jobs['is_active'] = dl_res.apply(lambda x: x[1])

print("=== BƯỚC 11: TRƯỚC VÀ SAU KHI CHUẨN HÓA DEADLINE & IS_ACTIVE ===")
jobs[['deadline_raw', 'deadline', 'is_active']].head(5)


=== BƯỚC 11: TRƯỚC VÀ SAU KHI CHUẨN HÓA DEADLINE & IS_ACTIVE ===


,deadline_raw,deadline,is_active
0,2026-10-28T23:59:59+07:00,2026-10-28,False
1,2026-10-28T23:59:59+07:00,2026-10-28,False
2,2026-11-05T23:59:59+07:00,2026-11-05,True
3,2026-10-28T23:59:59+07:00,2026-10-28,False
4,2026-10-29T23:59:59+07:00,2026-10-29,False


## Bước 12: Khử trùng lặp nhiều lớp (Deduplication)

Phiên bản cũ chỉ loại tin có `company_name` và `title` **giống hệt**, nên bỏ sót:
- cùng một công ty nhưng tên viết khác (`MBBANK` ↔ `NGÂN HÀNG TMCP QUÂN ĐỘI – MBBANK`),
- tiêu đề chỉ khác mã tin trong ngoặc (`... (2025TD943)`),
- bài đăng lại với tiêu đề sửa nhẹ.

Phiên bản mới có 3 lớp:

| Lớp | Điều kiện | Nhãn lý do |
|---|---|---|
| 1 | Cùng *khóa công ty* (bỏ dấu, bỏ từ pháp lý) và cùng *khóa tiêu đề* | `exact` |
| 2 | Cùng tiêu đề, tên công ty "chứa nhau", mô tả giống ≥ 0,80 (cosine TF-IDF) | `company_name_variant` |
| 3 | Cùng công ty, mô tả ≥ 0,92, tiêu đề ≥ 0,85, cùng thành phố, cùng yêu cầu kinh nghiệm | `same_description` |

> **Lưu ý quan trọng:** chỉ so mô tả là **không đủ**. Nhiều tin cùng công ty chia sẻ phần giới thiệu/phúc lợi giống hệt nhau, nên khi chỉ dùng ngưỡng 0,92 thuật toán đã gộp nhầm các vị trí khác nhau (vd. *Business Analyst* với *Cloud Engineer*). Vì vậy lớp 3 bắt buộc phải kèm điều kiện tiêu đề, thành phố và kinh nghiệm.

Trong mỗi nhóm trùng, giữ tin có **hạn chót xa nhất** và **mô tả dài nhất**. Bảng `df_dropped` lưu lại mọi tin bị loại để đối chiếu.

In [ ]:
from typing import Dict, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

_LEGAL_TOKENS = {
    "cong", "ty", "cty", "tnhh", "mtv", "co", "phan", "cp", "jsc", "llc", "inc", "corp", "ltd",
    "company", "limited", "joint", "stock", "group", "tap", "doan", "ngan", "hang", "tmcp",
    "chi", "nhanh", "vietnam", "viet", "nam", "the", "and", "va", "mot", "thanh", "vien",
}


def company_tokens(name: str) -> frozenset:
    """Tập token đặc trưng của tên công ty (bỏ dấu, bỏ từ pháp lý/chung chung)."""
    toks = re.findall(r"[a-z0-9]+", strip_accents(name))
    core = [t for t in toks if t not in _LEGAL_TOKENS]
    return frozenset(core or toks)


def company_key(name: str) -> str:
    return " ".join(sorted(company_tokens(name)))


def companies_compatible(a: frozenset, b: frozenset) -> bool:
    """Cùng một công ty nếu token tên này nằm trọn trong tên kia (MBBANK ⊂ Quân Đội MBBANK)."""
    return bool(a) and bool(b) and (a <= b or b <= a)


def title_key(title: str) -> str:
    """Tiêu đề chuẩn hóa: bỏ dấu, bỏ mã tin trong ngoặc và ký tự không phải chữ-số."""
    t = strip_accents(title)
    t = re.sub(r"\([^)]*\d[^)]*\)", " ", t)  # (2025td943), (ma tin 123)
    return " ".join(re.findall(r"[a-z0-9+#.]+", t))


def _jaccard(a: str, b: str) -> float:
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / len(sa | sb) if (sa | sb) else 0.0


class _UnionFind:
    def __init__(self, n: int):
        self.p = list(range(n))

    def find(self, x: int) -> int:
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a: int, b: int):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra


def deduplicate_jobs(
    jobs: pd.DataFrame,
    title_block_sim: float = 0.80,
    company_block_sim: float = 0.92,
    title_jaccard: float = 0.85,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Khử trùng lặp 3 lớp:
      1. Cùng (khóa công ty, khóa tiêu đề)                                   -> trùng chính xác
      2. Cùng tiêu đề + công ty tương thích + mô tả giống >= title_block_sim -> biến thể tên công ty
      3. Cùng công ty + mô tả giống >= company_block_sim                     -> đổi tiêu đề nhẹ
    Giữ bản có hạn nộp xa nhất, mô tả dài nhất. Trả về (df đã khử, báo cáo các bản bị loại).
    """
    df = jobs.reset_index(drop=True).copy()
    n = len(df)
    df["_ck"] = df["company_name"].map(company_key)
    df["_ct"] = df["company_name"].map(company_tokens)
    df["_tk"] = df["title"].map(title_key)
    desc = df["description_clean"].fillna("").astype(str)
    df["_len"] = desc.str.len()

    X = TfidfVectorizer(sublinear_tf=True).fit_transform(desc)
    uf = _UnionFind(n)
    reason: Dict[int, str] = {}

    def link(i: int, j: int, why: str):
        uf.union(i, j)
        reason.setdefault(i, why)
        reason.setdefault(j, why)

    # Lớp 1: trùng chính xác
    for _, idx in df.groupby(["_ck", "_tk"]).indices.items():
        for j in idx[1:]:
            link(idx[0], j, "exact")

    # Lớp 2: cùng tiêu đề, công ty tương thích, mô tả giống
    for tk, idx in df.groupby("_tk").indices.items():
        if len(idx) < 2 or len(tk) < 3:
            continue
        sim = linear_kernel(X[idx])
        for a in range(len(idx)):
            for b in range(a + 1, len(idx)):
                i, j = idx[a], idx[b]
                if sim[a, b] >= title_block_sim and companies_compatible(df.at[i, "_ct"], df.at[j, "_ct"]):
                    link(i, j, "company_name_variant")

    # Lớp 3: cùng công ty, mô tả gần như y hệt VÀ tiêu đề giống, cùng thành phố, cùng yêu cầu kinh nghiệm.
    # (Chỉ so mô tả là không đủ: nhiều tin cùng công ty chia sẻ phần giới thiệu/phúc lợi giống hệt nhau.)
    city = df["city"] if "city" in df.columns else pd.Series([""] * n)
    exp = df["experience_min"] if "experience_min" in df.columns else pd.Series([0.0] * n)
    for _, idx in df.groupby("_ck").indices.items():
        if len(idx) < 2:
            continue
        sim = linear_kernel(X[idx])
        for a in range(len(idx)):
            for b in range(a + 1, len(idx)):
                i, j = idx[a], idx[b]
                if (sim[a, b] >= company_block_sim
                        and _jaccard(df.at[i, "_tk"], df.at[j, "_tk"]) >= title_jaccard
                        and city[i] == city[j]
                        and (exp[i] == exp[j] or (pd.isna(exp[i]) and pd.isna(exp[j])))):
                    link(i, j, "same_description")

    df["_grp"] = [uf.find(i) for i in range(n)]
    df["_deadline"] = df["deadline"].fillna("") if "deadline" in df.columns else ""
    ordered = df.sort_values(by=["_grp", "_deadline", "_len"], ascending=[True, False, False])
    kept = ordered.drop_duplicates("_grp", keep="first")
    dropped = df.loc[~df.index.isin(kept.index)].copy()
    rep = kept.set_index("_grp")
    dropped["kept_job_id"] = dropped["_grp"].map(rep["job_id"])
    dropped["kept_company"] = dropped["_grp"].map(rep["company_name"])
    dropped["reason"] = [reason.get(i, "duplicate") for i in dropped.index]
    report = dropped[["job_id", "company_name", "title", "kept_job_id", "kept_company", "reason"]].reset_index(drop=True)

    result = df.loc[kept.sort_index().index].drop(columns=[c for c in df.columns if c.startswith("_")])
    return result.reset_index(drop=True), report


old_count = len(jobs)
jobs, df_dropped = deduplicate_jobs(jobs)

print('=' * 75)
print('BÁO CÁO KHỬ TRÙNG LẶP (DEDUPLICATION AUDIT REPORT)')
print(f'- Tổng số bản ghi trước khi khử trùng: {old_count:,} dòng')
print(f'- Số bản ghi bị loại bỏ:               {len(df_dropped):,} dòng')
print(f'- Phân loại theo lý do:                {df_dropped["reason"].value_counts().to_dict()}')
print(f'- Số lượng bản ghi duy nhất còn lại:   {len(jobs):,} dòng')
print('=' * 75)

print('\nMẪU CÁC BẢN GHI BỊ LOẠI (đối chiếu với bản được giữ lại):')
display(df_dropped.groupby('reason').head(4).sort_values('reason')[['reason', 'company_name', 'title', 'kept_company']])

## Bước 12b: Gắn cờ tin thuộc ngành IT (`is_it`)

Dữ liệu cào từ TopCV/VietnamWorks **không chỉ có việc IT**: có cả *Graphic Designer*, *Nhân viên kinh doanh*, *Kế toán*... Các tin này làm nhiễu top-N gợi ý và lệch trọng số IDF của TF-IDF.

Thay vì xóa, ta **gắn cờ `is_it`** để mỗi bước sau (EDA, recommender) tự chọn có dùng hay không. Một tin được coi là IT khi:
1. Tiêu đề chứa từ khóa nghề IT (*developer, engineer, tester, data, devops, lập trình, phần mềm...*), **hoặc**
2. Có ít nhất 2 kỹ năng công nghệ lõi (Python, SQL, Docker, AWS...).

In [ ]:
from typing import Iterable

# Chức danh/từ khóa đặc trưng của nghề IT
IT_TITLE_PATTERN = re.compile(
    r"(develop|engineer|lập trình|programmer|software|phần mềm|backend|back-end|front-?end|"
    r"full-?stack|devops|devsecops|\bsre\b|tester|\bqa\b|\bqc\b|kiểm thử|automation|\bdata\b|"
    r"dữ liệu|\bai\b|\bml\b|machine learning|nlp|deep learning|computer vision|blockchain|web3|"
    r"solidity|security|an ninh mạng|an toàn thông tin|pentest|\bsoc\b|dfir|system|hệ thống|"
    r"network|mạng|sysadmin|infrastructure|cloud|\baws\b|azure|database|\bdba\b|\bdev\b|\bit\b|"
    r"công nghệ thông tin|cntt|\bbi\b|business analyst|phân tích nghiệp vụ|\bba\b|product owner|"
    r"scrum|tech lead|technical|solution|architect|\berp\b|\bsap\b|abap|odoo|salesforce|"
    r"\.net|java|python|php|golang|\bgo\b|node|react|angular|vue|flutter|android|\bios\b|mobile|"
    r"unity|embedded|firmware|\biot\b|fpga|\bsql\b|\betl\b|crawler|scraper|helpdesk|help desk|"
    r"it support|technical support|ui/ux|ux/ui|\bux\b|\bui\b|bridge se|\bbrse\b|\bse\b|"
    r"wordpress|shopify|magento|\brpa\b|low-?code|power bi|tableau|analyst|\bcto\b|"
    r"triển khai|game dev|game programmer|technical artist|ux/?ui|product manager|kỹ sư|it project|it - phần mềm)",
    re.IGNORECASE,
)

# Kỹ năng công nghệ lõi: có >= 2 kỹ năng này thì coi là việc IT dù tiêu đề mơ hồ
CORE_IT_SKILLS = {
    "python", "java", "javascript", "typescript", "c++", "c#", "php", "golang", "ruby", "rust",
    "kotlin", "swift", "dart", "sql", "mysql", "postgresql", "mongodb", "redis", "sql server",
    "react", "vue", "angular", "node.js", ".net", "spring boot", "django", "laravel", "flutter",
    "docker", "kubernetes", "aws", "azure", "gcp", "linux", "git", "ci/cd", "html/css",
    "rest api", "microservices", "selenium", "machine learning", "deep learning", "apache spark",
    "apache kafka", "apache airflow", "etl", "android", "ios", "terraform", "jenkins",
}


def _skill_list(skills: object) -> list:
    if skills is None or isinstance(skills, float):  # None hoặc NaN
        return []
    if isinstance(skills, str):
        return [s.strip().lower() for s in skills.split(",") if s.strip()]
    return [str(s).strip().lower() for s in skills if str(s).strip()]


def is_it_job(title: str, required_skills: object = None, min_core_skills: int = 2) -> bool:
    """True nếu tiêu đề mang tính IT hoặc có >= min_core_skills kỹ năng công nghệ lõi."""
    if title and IT_TITLE_PATTERN.search(str(title)):
        return True
    return len(set(_skill_list(required_skills)) & CORE_IT_SKILLS) >= min_core_skills


jobs['is_it'] = jobs.apply(lambda r: is_it_job(r['title'], r['required_skills']), axis=1)
print(f"Tin thuộc ngành IT: {int(jobs['is_it'].sum()):,}/{len(jobs):,} ({jobs['is_it'].mean():.1%})")
print("\nVí dụ tin KHÔNG thuộc IT (bị gắn is_it = False):")
display(jobs.loc[~jobs['is_it'], ['title', 'company_name']].head(8))
print(f"Hoàn tất khử trùng lặp! Tập dữ liệu còn lại: {len(jobs):,} bản ghi duy nhất.")

## Bước 13: Xuất bảng kết quả cuối cùng ra CSV

Lựa chọn các cột chuẩn đầu ra và lưu vào thư mục `data/notebook/content_base/jobs_clean.csv`.

In [ ]:
final_columns = [
    'job_id',
    'company_name',
    'title',
    'description_clean',
    'required_skills',
    'preferred_skills',
    'job_tags',
    'experience_min',
    'experience_max',
    'location',
    'city',
    'salary_min',
    'salary_max',
    'employment_type',
    'work_mode',
    'deadline',
    'is_active',
    'is_it',
    'job_url',
    'company_logo_url'
]

df_final = jobs[final_columns].copy()

# Định dạng cột kỹ năng thành chuỗi nối dấu phẩy để lưu CSV
df_final_csv = df_final.copy()
df_final_csv['required_skills'] = df_final_csv['required_skills'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
df_final_csv['preferred_skills'] = df_final_csv['preferred_skills'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
df_final_csv['job_tags'] = df_final_csv['job_tags'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')

# 1. Lưu đè vào data/notebook/content_base/jobs_clean.csv (cho các Notebook EDA & CBF)
nb_output_dir = os.path.join(project_root, 'data', 'notebook', 'content_base')
os.makedirs(nb_output_dir, exist_ok=True)
output_csv_path = os.path.join(nb_output_dir, 'jobs_clean.csv')
df_final_csv.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

# 2. Đồng bộ lưu vào data/silver/ (cho ứng dụng Streamlit app/Home.py)
silver_dir = os.path.join(project_root, 'data', 'silver')
os.makedirs(silver_dir, exist_ok=True)
silver_csv_path = os.path.join(silver_dir, 'jobs_cleaned.csv')
silver_parquet_path = os.path.join(silver_dir, 'jobs_cleaned.parquet')

df_final_csv.to_csv(silver_csv_path, index=False, encoding='utf-8-sig')
try:
    df_final_csv.to_parquet(silver_parquet_path, index=False)
except Exception:
    pass

print('=' * 75)
print('✅ ĐÃ CHUẨN HÓA VÀ ĐỒNG BỘ 100% DỮ LIỆU VÀO TOÀN BỘ HỆ THỐNG:')
print(f'1. [Notebooks EDA & CBF]  : {output_csv_path}')
print(f'2. [Streamlit App Engine] : {silver_parquet_path}')
print(f'3. [Silver CSV Storage]   : {silver_csv_path}')
print(f'📊 Quy mô: {len(df_final_csv):,} việc làm sạch x {df_final_csv.shape[1]} thuộc tính (đầy đủ logo, url & cờ is_it).')
print('=' * 75)
df_final_csv.head(3)